# Few-Shot Transfer Learning for TBI Length-of-Stay — **corrected**

This notebook is a corrected duplicate of `transfer_learning_experiment.ipynb`.

**What changed (Fix #1):** the transfer-learning models now use the **full harmonized
feature set** — `age`, `gcs` (numeric) plus `sex`, `hr_cat`, `rr_cat`, `sbp_cat`, `MOI`
(one-hot encoded) — instead of only `age, sex, gcs`. A **single `OneHotEncoder` is fit
across all cohorts**, so the source-pretrained and target-fine-tuned models share an
identical feature space (required for warm-start fine-tuning).

**Also corrected:** repetitions are now genuine — both the few-shot resample **and** the
estimator are seeded *per iteration* (`seed = 42 + rep`), so the reported mean ± SD and the
Wilcoxon tests reflect real variability (previously all repetitions were identical).

See `METHODS.md` for the full description.

In [1]:
import os, copy, warnings, itertools
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils import compute_sample_weight
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, cumulative_dynamic_auc
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

NUMERIC = ["age", "gcs"]
CATEG   = ["sex", "hr_cat", "rr_cat", "sbp_cat", "MOI"]
RATIOS  = [0.05, 0.10, 0.20]
N_REP   = 20

In [2]:
# Load cohorts (only those present; Florida is available on request from the author)
candidate = {"India": "india_clean.csv", "Jordan": "jordan_clean.csv",
             "Florida": "florida_clean.csv", "California": "california_clean.csv"}
registry = {}
for name, fname in candidate.items():
    if os.path.exists(fname):
        df = pd.read_csv(fname)
        df["los"] = df["los"].clip(lower=1)   # strictly positive survival times
        registry[name] = df
print("Loaded cohorts:", {k: len(v) for k, v in registry.items()})

Loaded cohorts: {'India': 7978, 'Jordan': 112, 'Florida': 263, 'California': 583}


In [3]:
# ---- unified feature space ------------------------------------------------
def cat_frame(df):
    """Normalize categorical columns so categories are consistent across cohorts."""
    f = pd.DataFrame(index=df.index)
    f["sex"] = pd.to_numeric(df["sex"], errors="coerce").round().astype("Int64").astype(str)
    for c in ["hr_cat", "rr_cat", "sbp_cat", "MOI"]:
        f[c] = df[c].astype(str)
    return f

# Fit ONE encoder across every available cohort -> shared one-hot columns
all_cat = pd.concat([cat_frame(df) for df in registry.values()], ignore_index=True)
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(all_cat)
print(f"Feature space: {len(NUMERIC)} numeric + "
      f"{len(encoder.get_feature_names_out())} one-hot "
      f"= {len(NUMERIC) + len(encoder.get_feature_names_out())} columns")

def to_X(df):
    num = df[NUMERIC].to_numpy(dtype=float)
    cat = encoder.transform(cat_frame(df))
    return np.hstack([num, cat])

def to_y(df):
    return np.array([(bool(e), t) for e, t in zip(df["event"], df["los"])],
                    dtype=[("event", bool), ("time", float)])

def cindex(y, risk):
    return concordance_index_censored(y["event"], y["time"], risk)[0]

Feature space: 2 numeric + 19 one-hot = 21 columns


In [4]:
# Pretrain each source cohort ONCE (fixed seed) and reuse via deep-copy.
# scikit-survival GBSA is slow on the large India cohort, so re-pretraining the
# source every repetition is wasteful; repetition variability legitimately comes
# from the target resample + fine-tuning, not from re-pretraining the source.
source_models = {}
for name in registry:
    source_models[name] = GradientBoostingSurvivalAnalysis(
        n_estimators=100, warm_start=True, random_state=42).fit(to_X(registry[name]), to_y(registry[name]))
    print("pretrained", name)

pretrained India
pretrained Jordan
pretrained Florida


pretrained California


In [5]:
def run_pair(source, target, ratios=RATIOS, n_rep=N_REP):
    """Baseline vs Standard TL vs Weighted TL, few-shot at each ratio, n_rep repetitions."""
    X_tgt, y_tgt = to_X(registry[target]), to_y(registry[target])
    base_src = source_models[source]
    rows = []
    for rep in range(n_rep):
        seed = 42 + rep
        X_tr, X_te, y_tr, y_te = train_test_split(
            X_tgt, y_tgt, test_size=0.2, random_state=seed)
        y_ev = y_tr["event"].astype(int)
        for ratio in ratios:
            sss = StratifiedShuffleSplit(n_splits=1, train_size=ratio, random_state=seed)
            idx, _ = next(sss.split(X_tr, y_ev))
            X_fs, y_fs = X_tr[idx], y_tr[idx]

            # (a) baseline: target-only
            mb = GradientBoostingSurvivalAnalysis(n_estimators=100, random_state=seed)
            mb.fit(X_fs, y_fs)
            c_base = cindex(y_te, mb.predict(X_te))

            # (b) standard TL: 100 source stages + 50 target stages (warm start)
            ms = copy.deepcopy(base_src); ms.set_params(n_estimators=150)
            ms.fit(X_fs, y_fs)
            c_std = cindex(y_te, ms.predict(X_te))

            # (c) weighted TL: deaths up-weighted 5x during fine-tuning
            mw = copy.deepcopy(base_src); mw.set_params(n_estimators=150)
            sw = compute_sample_weight(class_weight={1: 5, 0: 1}, y=y_fs["event"].astype(int))
            mw.fit(X_fs, y_fs, sample_weight=sw)
            c_wt = cindex(y_te, mw.predict(X_te))

            rows.append({"source": source, "target": target, "rep": rep, "ratio": ratio,
                         "baseline": c_base, "standard": c_std, "weighted": c_wt})
    return pd.DataFrame(rows)

In [6]:
# Run all ordered source -> target pairs
pairs = list(itertools.permutations(registry.keys(), 2))
all_results = []
for s, t in pairs:
    df = run_pair(s, t)
    all_results.append(df)
    r5 = df[df.ratio == 0.05]
    print(f"{s:>10} -> {t:<10} [5%]  base={r5.baseline.mean():.3f}  "
          f"std={r5.standard.mean():.3f}  wt={r5.weighted.mean():.3f}")

results = pd.concat(all_results, ignore_index=True)
results.to_csv("fixed_results_raw.csv", index=False)
results.head()

     India -> Jordan     [5%]  base=0.601  std=0.806  wt=0.807


     India -> Florida    [5%]  base=0.565  std=0.504  wt=0.488


     India -> California [5%]  base=0.622  std=0.821  wt=0.827


    Jordan -> India      [5%]  base=0.741  std=0.719  wt=0.724


    Jordan -> Florida    [5%]  base=0.565  std=0.522  wt=0.518


...

   Florida -> California [5%]  base=0.622  std=0.598  wt=0.616


California -> India      [5%]  base=0.741  std=0.736  wt=0.740


California -> Jordan     [5%]  base=0.601  std=0.844  wt=0.845


California -> Florida    [5%]  base=0.565  std=0.526  wt=0.514


,source,target,rep,ratio,baseline,standard,weighted
0,India,Jordan,0,0.05,0.317568,0.891892,0.959459
1,India,Jordan,0,0.10,0.790541,0.837838,0.837838
2,India,Jordan,0,0.20,0.878378,0.959459,0.959459
3,India,Jordan,1,0.05,0.500000,0.877049,0.877049
4,India,Jordan,1,0.10,0.581967,0.852459,0.852459


In [7]:
# Summary: mean +/- SD of C-index over repetitions
summary = (results.groupby(["source", "target", "ratio"])[["baseline", "standard", "weighted"]]
                  .agg(["mean", "std"]).round(3))
summary.to_csv("fixed_results_summary.csv")
summary

baseline        standard        weighted       
                                mean    std     mean    std     mean    std
source     target     ratio                                                
California Florida    0.05     0.565  0.100    0.526  0.124    0.514  0.129
                      0.10     0.616  0.094    0.549  0.116    0.545  0.117
                      0.20     0.554  0.121    0.538  0.104    0.558  0.118
           India      0.05     0.741  0.019    0.736  0.014    0.740  0.017
                      0.10     0.759  0.013    0.750  0.011    0.750  0.013
                      0.20     0.770  0.014    0.756  0.012    0.756  0.014
           Jordan     0.05     0.601  0.208    0.844  0.080    0.845  0.081
                      0.10     0.663  0.241    0.849  0.110    0.846  0.082
                      0.20     0.746  0.186    0.834  0.110    0.832  0.109
Florida    California 0.05     0.622  0.173    0.598  0.189    0.616  0.185
                      0.10     0.611  0.135    0.608  0.151    0.629  0.157
                      0.20     0.613  0.158    0.584  0.144    0.616  0.150
           India      0.05     0.741  0.019    0.722  0.021    0.737  0.022
                      0.10     0.759  0.013    0.738  0.018    0.750  0.015
                      0.20     0.770  0.014    0.746  0.017    0.758  0.015
           Jordan     0.05     0.601  0.208    0.629  0.227    0.656  0.215
                      0.10     0.663  0.241    0.689  0.219    0.670  0.213
                      0.20     0.746  0.186    0.722  0.206    0.706  0.176
India      California 0.05     0.622  0.173    0.821  0.081    0.827  0.072
                      0.10     0.611  0.135    0.820  0.056    0.811  0.056
                      0.20     0.613  0.158    0.826  0.056    0.816  0.062
           Florida    0.05     0.565  0.100    0.504  0.121    0.488  0.117
                      0.10     0.616  0.094    0.533  0.125    0.509  0.112
                      0.20     0.554  0.121    0.502  0.120    0.514  0.136
           Jordan     0.05     0.601  0.208    0.806  0.139    0.807  0.152
                      0.10     0.663  0.241    0.793  0.153    0.793  0.153
                      0.20     0.746  0.186    0.806  0.140    0.825  0.145
Jordan     California 0.05     0.622  0.173    0.670  0.136    0.672  0.144
                      0.10     0.611  0.135    0.692  0.120    0.691  0.121
                      0.20     0.613  0.158    0.665  0.119    0.705  0.110
           Florida    0.05     0.565  0.100    0.522  0.134    0.518  0.139
                      0.10     0.616  0.094    0.562  0.153    0.556  0.140
                      0.20     0.554  0.121    0.526  0.108    0.531  0.113
           India      0.05     0.741  0.019    0.719  0.018    0.724  0.021
                      0.10     0.759  0.013    0.732  0.015    0.737  0.012
                      0.20     0.770  0.014    0.743  0.016    0.743  0.014

In [8]:
# Wilcoxon signed-rank tests across repetitions (per pair x ratio)
wil = []
for (s, t, ratio), g in results.groupby(["source", "target", "ratio"]):
    def w(a, b):
        try:    return round(wilcoxon(g[a], g[b]).pvalue, 4)
        except Exception: return np.nan
    wil.append({"source": s, "target": t, "ratio": ratio,
                "p_base_vs_std": w("baseline", "standard"),
                "p_base_vs_wt":  w("baseline", "weighted"),
                "p_std_vs_wt":   w("standard", "weighted")})
wil_df = pd.DataFrame(wil)
wil_df.to_csv("fixed_results_wilcoxon.csv", index=False)
wil_df

,source,target,ratio,p_base_vs_std,p_base_vs_wt,p_std_vs_wt
0,California,Florida,0.05,0.2455,0.1231,0.4955
1,California,Florida,0.10,0.0192,0.0121,0.6791
2,California,Florida,0.20,0.3341,0.9854,0.1221
3,California,India,0.05,0.1231,0.5217,0.1054
4,California,India,0.10,0.0000,0.0007,0.8124
5,California,India,0.20,0.0000,0.0000,0.7012
6,California,Jordan,0.05,0.0000,0.0000,0.6547
7,California,Jordan,0.10,0.0005,0.0025,0.3441
8,California,Jordan,0.20,0.0176,0.0279,0.9165
9,Florida,California,0.05,0.2943,0.7562,0.4328


In [ ]:
# Figure 4: one 3D bar chart per source cohort (a separate figure/PNG for each)
# axes: Target Dataset (x, method labels perpendicular) x Few-shot Ratio (y) x C-index (z)
from mpl_toolkits.mplot3d import Axes3D  # noqa
from matplotlib.patches import Patch
m = summary.xs("mean", axis=1, level=1).reset_index()   # source,target,ratio,baseline,standard,weighted
ORDER = ["India", "Jordan", "Florida", "California"]
sources = [x for x in ORDER if x in m["source"].unique()]
METHODS = [("baseline", "Baseline", "#1f77b4"),
           ("standard", "Standard TL", "#ff7f0e"),
           ("weighted", "Weighted TL", "#2ca02c")]
RLAB = [5, 10, 20]; GROUP_STEP = len(METHODS) + 1; DX, DY = 0.75, 0.55
for src in sources:
    fig = plt.figure(figsize=(9.5, 8.5), dpi=300)
    ax = fig.add_subplot(111, projection="3d")
    targets = [t for t in ORDER if t != src and t in m["target"].unique()]
    xticks, xlabels = [], []
    for ti, tg in enumerate(targets):
        for mi, (col, mlab, mc) in enumerate(METHODS):
            xpos = ti * GROUP_STEP + mi
            for ri, r in enumerate(RATIOS):
                row = m[(m["source"] == src) & (m["target"] == tg) & (np.isclose(m["ratio"], r))]
                if row.empty:
                    continue
                ax.bar3d(xpos, ri - DY / 2, 0, DX, DY, float(row[col].values[0]),
                         color=mc, shade=True, edgecolor="white", linewidth=0.2)
            xticks.append(xpos + DX / 2); xlabels.append(mlab)
    ax.set_title(f"Source Dataset: {src}", fontsize=18, pad=10)
    ax.set_zlim(0, 1.0); ax.set_zlabel("C-index", fontsize=15, labelpad=10)
    ax.set_yticks(range(len(RATIOS))); ax.set_yticklabels(RLAB)
    ax.set_ylabel("Few-shot Ratio (%)", fontsize=15, labelpad=14)
    ax.tick_params(axis="y", labelsize=13); ax.tick_params(axis="z", labelsize=13)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels, fontsize=13, rotation=60, ha="right", va="top", rotation_mode="anchor")
    ax.set_xlabel("")   # target-group brackets/names added in post (drawio)
    ax.view_init(elev=20, azim=-60)
    ax.set_box_aspect((len(targets) * GROUP_STEP, len(RATIOS) + 1, 6))
    handles = [Patch(color=mc, label=mlab) for _, mlab, mc in METHODS]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize=14,
               bbox_to_anchor=(0.5, 0.05))
    fig.subplots_adjust(left=0.02, right=0.80, bottom=0.16, top=0.94)
    fig.savefig(f"fig4_cindex_{src}.png", dpi=300)
    plt.show()

In [ ]:
# Figure 5: dynamic AUC, one figure per (source->target, few-shot ratio),
# comparing Baseline vs Standard TL vs Weighted TL (12 pairs x 3 ratios = 36 figures).
import os
os.makedirs("results_auc", exist_ok=True)

def dyn_auc(y_all, risk, times):
    try:
        arr, _ = cumulative_dynamic_auc(y_all, y_all, risk, times)
        return np.asarray(arr)
    except Exception:
        out = []
        for tt in times:
            try:
                _, a = cumulative_dynamic_auc(y_all, y_all, risk, times=tt); out.append(a)
            except Exception:
                out.append(np.nan)
        return np.asarray(out)

for s, t in pairs:
    X_tgt, y_tgt = to_X(registry[t]), to_y(registry[t])
    X_tr, X_te, y_tr, y_te = train_test_split(X_tgt, y_tgt, test_size=0.2, random_state=42)
    y_ev = y_tr["event"].astype(int)
    tmin = max(y_te["time"].min(), y_tgt["time"].min())
    tmax = min(y_te["time"].max(), y_tgt["time"].max()) - 1e-6
    times = np.linspace(tmin, tmax, 50)
    for ratio in RATIOS:
        idx, _ = next(StratifiedShuffleSplit(1, train_size=ratio, random_state=42).split(X_tr, y_ev))
        X_fs, y_fs = X_tr[idx], y_tr[idx]
        mb = GradientBoostingSurvivalAnalysis(n_estimators=100, random_state=42).fit(X_fs, y_fs)
        ms = copy.deepcopy(source_models[s]); ms.set_params(n_estimators=150); ms.fit(X_fs, y_fs)
        mw = copy.deepcopy(source_models[s]); mw.set_params(n_estimators=150)
        sw = compute_sample_weight(class_weight={1: 5, 0: 1}, y=y_fs["event"].astype(int))
        mw.fit(X_fs, y_fs, sample_weight=sw)
        auc_b = dyn_auc(y_tgt, mb.predict(X_tgt), times)
        auc_s = dyn_auc(y_tgt, ms.predict(X_tgt), times)
        auc_w = dyn_auc(y_tgt, mw.predict(X_tgt), times)
        plt.figure(figsize=(8, 5), dpi=300)
        plt.plot(times, auc_b, marker="o", ms=4, color="#1f77b4", label="Baseline")
        plt.plot(times, auc_s, marker="s", ms=4, color="#ff7f0e", label="Standard TL")
        plt.plot(times, auc_w, marker="^", ms=4, color="#2ca02c", label="Weighted TL")
        plt.axhline(0.5, ls="--", color="gray", lw=1)
        plt.title(f"Few-Shot Transfer @ {int(ratio*100)}%: {s} → {t}", fontsize=13)
        plt.xlabel("Days", fontsize=12); plt.ylabel("Dynamic AUC", fontsize=12)
        plt.ylim(0.3, 1.0); plt.grid(True, alpha=0.3); plt.legend(fontsize=11); plt.tight_layout()
        plt.savefig(f"results_auc/auc_{s}_to_{t}_{int(ratio*100)}pct.png", dpi=300)
        plt.show()
print("Saved 36 dynamic-AUC figures to results_auc/")